# OU Leader to Territory Mapping: Approach Comparison

## Executive Summary

A teammate asked how to reliably map OU leaders to their territories for FY2026. Their current approach uses `employeeaccountassignments` to derive subregion/subsegment from TPIDs, but this produces noise -- some leaders show SMEC TPIDs they do not actually own.

This notebook compares two approaches with real data:

- **Option 1 (Recommended):** Map via territory/deployment hierarchy tables -- reflects structural ownership.
- **Option 2:** Map via `employeeaccountassignments` with proper filters -- reflects compensation-level account touchpoints.

**Fiscal Year:** FY2026 | **Segment Filter:** Sales Leaders

---

## 1. Setup and Connection

### 1.1 Import Libraries

Uses the same connection pattern from `kusto_app/kusto_connection.py` -- AAD authentication with Kusto Explorer token cache as fallback.

> **Note:** If auth fails with error 530033 (device compliance), open Kusto Explorer first, connect to the cluster, run any query, then retry here. The cached token will be reused.

In [1]:
"""1.1 Import libraries and configure display settings."""

import os
import re
import logging
from typing import Optional

import pandas as pd
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.exceptions import KustoServiceError

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 50)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

logger.info("Libraries loaded successfully.")

INFO: Libraries loaded successfully.


In [2]:
"""1.2 Configuration -- cluster, database, and fiscal year."""

KUSTO_CLUSTER = "https://mscadx.westus2.kusto.windows.net"
KUSTO_DATABASE = "WWICLake"
FISCAL_YEAR = "2026"

logger.info("Cluster:  %s", KUSTO_CLUSTER)
logger.info("Database: %s", KUSTO_DATABASE)
logger.info("FY:       %s", FISCAL_YEAR)

INFO: Cluster:  https://mscadx.westus2.kusto.windows.net
INFO: Database: WWICLake
INFO: FY:       2026


In [3]:
"""1.3 Establish connection to Kusto via Azure CLI authentication.

Prerequisite: run 'az login --tenant microsoft.onmicrosoft.com' in terminal.
Uses the same proven pattern from kusto_data_extractor.py.
"""

from azure.kusto.data.helpers import dataframe_from_result_table

# Create connection using Azure CLI auth
kcsb = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER)
client = KustoClient(kcsb)

# Verify connection
test_query = "print test='connection_successful', timestamp=now()"
response = client.execute(KUSTO_DATABASE, test_query)
result_df = dataframe_from_result_table(response.primary_results[0])

logger.info("CONNECTION SUCCESSFUL")
logger.info("  User: rodolfolerma@microsoft.com")
logger.info("  Test: %s", result_df.iloc[0]["test"])
logger.info("  Time: %s", result_df.iloc[0]["timestamp"])

INFO: AzureCliCredential.get_token succeeded
INFO: CONNECTION SUCCESSFUL
INFO:   User: rodolfolerma@microsoft.com
INFO:   Test: connection_successful
INFO:   Time: 2026-02-24 16:19:42.539054700+00:00


In [4]:
"""1.4 Helper: execute read-only KQL queries.

Includes the same safety regex from kusto_connection.py to block
any write/modify operations before they reach the cluster.
Uses dataframe_from_result_table for correct column mapping.
"""

BLOCKED_REGEX = re.compile(
    r"\.drop\s|\.delete\s|\.purge\s|\.clear\s|\.create\s|"
    r"\.alter\s|\.append\s|\.set\s|\.set-or-append\s|"
    r"\.set-or-replace\s|\.replace\s|\.ingest\s|\.move\s|"
    r"\.rename\s|\.execute\s+database\s+script",
    re.IGNORECASE,
)


def run_kql(query: str, database: str = KUSTO_DATABASE) -> pd.DataFrame:
    """Execute a read-only KQL query and return results as a DataFrame.

    Args:
        query: KQL query string. Must be read-only.
        database: Target Kusto database name.

    Returns:
        pandas DataFrame with query results.

    Raises:
        ValueError: If query contains a blocked write/modify command.
        KustoServiceError: If the Kusto service returns an error.
        RuntimeError: If no active client connection exists.
    """
    if client is None:
        raise RuntimeError("No active Kusto connection. Run the connection cell first.")

    if BLOCKED_REGEX.search(query):
        raise ValueError("Blocked: write/modify operation detected in query.")

    response = client.execute(database, query)
    df = dataframe_from_result_table(response.primary_results[0])
    logger.info("Query returned %s rows x %s columns.", f"{len(df):,}", len(df.columns))
    return df


logger.info("Helper function ready.")

INFO: Helper function ready.


---
## 2. Identify Sales Leader PlanIDs

### 2.1 Query PlanIDs

This is the **common starting point** for both approaches -- get the PlanIDs for OU leaders from `plandetails` filtered on `BusinessSegmentName = 'Sales Leaders'` for FY2026.

In [23]:
"""2.1 Find Sales Leader PlanIDs from PlanDetails.

Strategy: Filter PlanDetails for plans whose PlanName or BucketCategory
indicates OU/Sales Leadership roles for the target fiscal year.
"""

# First, explore what plan names exist to find leader-specific plans
query_leader_plans = f"""
PlanDetails
| where DataFiscalYearID == {FISCAL_YEAR}
| where IsCurrent == true
| where PlanName has_any ('leader', 'manager', 'director', 'OU', 'Sales Unit')
    or BucketCategory has_any ('leader', 'manager')
| summarize
    BucketCategories = make_set(BucketCategory),
    BusinessSegments = make_set(BusinessSegmentName)
    by PlanID, PlanName
| order by PlanName asc
"""

logger.info("Searching for Sales Leader plan names in PlanDetails...")
df_leader_plans = run_kql(query_leader_plans)
logger.info("Found %d candidate plans", len(df_leader_plans))
df_leader_plans

INFO: Searching for Sales Leader plan names in PlanDetails...


INFO: Query returned 192 rows x 4 columns.
INFO: Found 192 candidate plans


,PlanID,PlanName,BucketCategories,BusinessSegments
0,723,FY26 10.50 Enterprise Commercial Executive Man...,"[All MS Cloud, Total ACR, All Security, AI Biz...",[Enterprise]
1,1321,FY26 10.51 Enterprise Commercial Executive Man...,"[All MS Cloud, Total ACR, All Security, AI Biz...",[Enterprise]
2,835,FY26 100.50 Field GSI/ESI PDM/PTS Manager,"[AI Biz Solutions, PI ACR, All Security Usage,...",[GPS]
3,853,FY26 101.50 Global GSI/ESI PDM/PTS Manager,"[AI Biz Solutions, All Security Usage, PI AI P...",[GPS]
4,1332,FY26 116.50 Field GSI/ESI Cloud + AI Platform ...,"[Migrate + Secure, PI ACR]",[GPS]
...,...,...,...,...
187,575,FY26 935.00 Curate AE/AM/PSE IC & Manager H1,[Total Curate],[Advertising]
188,576,FY26 936.00 Retail Media Curate AE/AM IC & Man...,"[Total Retail Media, Total Curate]",[Advertising]
189,1366,FY26 938.10 Curate Manager H2,[Total Curate],[Advertising]
190,1375,FY26 939.10 Gaming Manager H2,"[Total Retail Gaming Revenue, Gaming Pod]",[Advertising]


In [24]:
"""2.2 Narrow down to OU-level leader plans.

The 192 plans include all managers. Let us inspect the plan name patterns
to identify OU-level leaders specifically (e.g., Sales Unit Leaders, ATU/STU
leaders, area directors). We will also check the distinct PlanName keywords.
"""

# Show distinct keyword patterns in plan names
logger.info("Plan name keyword analysis:")
print("Plans containing 'Sales Unit':")
mask_su = df_leader_plans["PlanName"].str.contains("Sales Unit", case=False)
print(f"  Count: {mask_su.sum()}")
if mask_su.sum() > 0:
    for _, r in df_leader_plans[mask_su].iterrows():
        print(f"    PlanID={r['PlanID']}: {r['PlanName']}")

print("\nPlans containing 'ATU' or 'STU' or 'Area':")
mask_atu = df_leader_plans["PlanName"].str.contains("ATU|STU|Area", case=False)
print(f"  Count: {mask_atu.sum()}")
if mask_atu.sum() > 0:
    for _, r in df_leader_plans[mask_atu].head(20).iterrows():
        print(f"    PlanID={r['PlanID']}: {r['PlanName']}")

print("\nPlans containing 'Director':")
mask_dir = df_leader_plans["PlanName"].str.contains("Director", case=False)
print(f"  Count: {mask_dir.sum()}")
if mask_dir.sum() > 0:
    for _, r in df_leader_plans[mask_dir].head(20).iterrows():
        print(f"    PlanID={r['PlanID']}: {r['PlanName']}")

print("\nPlans containing 'Executive' or 'GM' or 'CVP':")
mask_exec = df_leader_plans["PlanName"].str.contains("Executive|\\bGM\\b|CVP", case=False)
print(f"  Count: {mask_exec.sum()}")
if mask_exec.sum() > 0:
    for _, r in df_leader_plans[mask_exec].head(20).iterrows():
        print(f"    PlanID={r['PlanID']}: {r['PlanName']}")

INFO: Plan name keyword analysis:


Plans containing 'Sales Unit':
  Count: 0

Plans containing 'ATU' or 'STU' or 'Area':
  Count: 3
    PlanID=1278: FY26 2400.00 OU/ Area/ Subregion/ Country Manager Leader - G1 - MS Federal
    PlanID=370: FY26 400.00 OU/ Area/ Subregion Leader - G1
    PlanID=959: FY26 401.00 OU/ Area/ Subregion Leader - G2

Plans containing 'Director':
  Count: 0

Plans containing 'Executive' or 'GM' or 'CVP':
  Count: 11
    PlanID=723: FY26 10.50 Enterprise Commercial Executive Manager
    PlanID=1321: FY26 10.51 Enterprise Commercial Executive Manager - US HLS
    PlanID=1221: FY26 2010.50 Enterprise Commercial Executive Manager - MS Federal
    PlanID=1342: FY26 2403.00 CVP/OU Leader - MS Federal
    PlanID=373: FY26 403.00 CVP/OU Leader
    PlanID=1378: FY26 403.01 CVP/OU Leader - US HLS
    PlanID=374: FY26 404.00 CVP/OU Leader (SMC)
    PlanID=728: FY26 600.50 Corporate Digital Account Executive Manager
    PlanID=956: FY26 605.50 Corporate Commercial Executive - Manager
    PlanID=935: FY26 8.

---
## 3. Get OU Leaders from Participation

### 3.1 Join PlanIDs to People

Join the discovered PlanIDs to `participation` and `person` tables to get the actual OU leader identities.

In [25]:
"""3.1 Retrieve OU leader people by joining Participation to Person.

Uses the OU-specific PlanIDs discovered in Section 2.
Join key: Participation.PersonnelNumber -> Person.PersonnelNumber.
"""

# Select OU Leader plans (400.xx and 403.xx family + federal variants)
ou_plan_ids = df_leader_plans[
    df_leader_plans["PlanName"].str.contains(
        r"OU[/ ]|CVP/OU", case=False, regex=True
    )
]["PlanID"].tolist()

logger.info("Selected %d OU Leader PlanIDs: %s", len(ou_plan_ids), ou_plan_ids)

plan_id_filter = ", ".join([str(p) for p in ou_plan_ids])

query_leaders = f"""
let LeaderPlanIDs = dynamic([{plan_id_filter}]);
Participation
| where DataFiscalYearID == {FISCAL_YEAR}
| where IsCurrent == true
| where PlanID in (LeaderPlanIDs)
| join kind=inner (
    Person
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
    | project PersonnelNumber, EmployeeName, EmployeeAlias, FullName
) on PersonnelNumber
| project PersonnelNumber, FullName, EmployeeAlias, PlanID, ParticipationID
| distinct *
| order by FullName asc
"""

logger.info("Querying OU Leaders from Participation + Person...")
df_leaders = run_kql(query_leaders)
df_leaders.head(20)

INFO: Selected 9 OU Leader PlanIDs: [284, 1341, 1278, 1342, 370, 959, 373, 1378, 374]
INFO: Querying OU Leaders from Participation + Person...
INFO: Query returned 109 rows x 5 columns.


,PersonnelNumber,FullName,EmployeeAlias,PlanID,ParticipationID
0,6399340,Agnes Heftberger,AHEFTBERGER,373,80426
1,228194,Ahmad El Dandachi,AHMADEL,370,80283
2,228194,Ahmad El Dandachi,AHMADEL,370,59322
3,6285531,Ahmed Hamzawy,HAMZAWYAHMED,284,73320
4,367417,Alaeddine Alaeddine,ALAKALA,959,60214
5,99570,Alberto Granados,ALBERTOG,370,74186
6,1189582,Alejandro Ferrer Machuca,1189582,284,73045
7,1186739,Ali Makke,ALIMAKKE,284,86985
8,1350041,Alon Haimovich,ALHAIMOV,370,65432
9,226504,Amr Kamel,AMRK,370,56529


In [26]:
"""3.2 Summary of discovered OU leaders."""

logger.info("Found %d OU Leader rows for FY%s", len(df_leaders), FISCAL_YEAR)
logger.info("Unique people: %d", df_leaders["PersonnelNumber"].nunique())
logger.info("Unique plans:  %d", df_leaders["PlanID"].nunique())
print(f"\nPlan distribution:")
plan_id_to_name = dict(zip(df_leader_plans["PlanID"], df_leader_plans["PlanName"]))
for pid, cnt in df_leaders["PlanID"].value_counts().items():
    name = plan_id_to_name.get(pid, "Unknown")
    print(f"  PlanID {pid} ({cnt} people): {name}")

INFO: Found 109 OU Leader rows for FY2026
INFO: Unique people: 105
INFO: Unique plans:  7



Plan distribution:
  PlanID 370 (44 people): FY26 400.00 OU/ Area/ Subregion Leader - G1
  PlanID 284 (22 people): FY26 162.00 GPS Field OU/Segment Leader
  PlanID 373 (19 people): FY26 403.00 CVP/OU Leader
  PlanID 959 (15 people): FY26 401.00 OU/ Area/ Subregion Leader - G2
  PlanID 1341 (4 people): FY26 165.00 GPS Field OU/Segment SME&C Leader
  PlanID 374 (4 people): FY26 404.00 CVP/OU Leader (SMC)
  PlanID 1378 (1 people): FY26 403.01 CVP/OU Leader - US HLS


---
## 4. Option 1: Territory/Deployment Hierarchy Approach (Recommended)

Map leaders to territories using the **structural territory hierarchy**. This gives you the territory a leader *owns* in the org structure, not just the accounts that touch their comp.

**Why this is better:**
- Reflects organizational ownership, not account-level rollups
- No noise from inherited/rollup SMEC accounts
- Single source of truth for territory structure

In [27]:
"""4.1 Option 1 query -- map leaders to territories via ParticipationTerritory + IncentiveTerritoryDetail.

Join path: Participation -> ParticipationTerritory (on ParticipationID)
           -> IncentiveTerritoryDetail (on IncentiveTerritoryID)

IncentiveTerritoryDetail uses DomainName/DomainValueName to describe
territory dimensions (e.g., SubRegion, Segment, Area, Country).
"""

query_option1 = f"""
let LeaderPlanIDs = dynamic([{plan_id_filter}]);
// Step 1: Get OU Leader participations
let Leaders =
    Participation
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
    | where PlanID in (LeaderPlanIDs)
    | join kind=inner (
        Person
        | where DataFiscalYearID == {FISCAL_YEAR}
        | where IsCurrent == true
        | project PersonnelNumber, FullName, EmployeeAlias
    ) on PersonnelNumber
    | project PersonnelNumber, FullName, EmployeeAlias, PlanID, ParticipationID;
// Step 2: Join to ParticipationTerritory for territory assignments
let LeaderTerritories =
    Leaders
    | join kind=inner (
        ParticipationTerritory
        | where DataFiscalYearID == {FISCAL_YEAR}
        | where IsCurrent == true
        | project ParticipationID, IncentiveTerritoryID, IncentiveTerritoryName
    ) on ParticipationID
    | project PersonnelNumber, FullName, EmployeeAlias, PlanID,
             IncentiveTerritoryID, IncentiveTerritoryName;
// Step 3: Enrich with IncentiveTerritoryDetail for domain-level breakdown
LeaderTerritories
| join kind=inner (
    IncentiveTerritoryDetail
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
    | project IncentiveTerritoryID, DomainName, DomainValueCode, DomainValueName
) on IncentiveTerritoryID
| project FullName, EmployeeAlias, PlanID,
         IncentiveTerritoryName, DomainName, DomainValueCode, DomainValueName
| distinct *
| order by FullName asc, DomainName asc
"""

logger.info("Running Option 1: Territory Hierarchy via ParticipationTerritory + IncentiveTerritoryDetail")
df_option1 = run_kql(query_option1)
logger.info("Option 1 returned %d rows", len(df_option1))
df_option1.head(30)

INFO: Running Option 1: Territory Hierarchy via ParticipationTerritory + IncentiveTerritoryDetail
INFO: Query returned 4,661 rows x 7 columns.
INFO: Option 1 returned 4661 rows


,FullName,EmployeeAlias,PlanID,IncentiveTerritoryName,DomainName,DomainValueCode,DomainValueName
0,Agnes Heftberger,AHEFTBERGER,373,AHEFTBERGER.SSG.1.20250701.20260630.1,SubRegion,72,Germany
1,Agnes Heftberger,AHEFTBERGER,373,AHEFTBERGER.SSG.1.20250701.20260630.1,SubRegion,86,Austria
2,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250701.20250830.1,End Customer Sub Segment,387,Upper Majors - Government
3,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250831.20260120.1,End Customer Sub Segment,396,Strategic - Commercial
4,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250701.20250830.1,End Customer Sub Segment,396,Strategic - Commercial
5,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250701.20250830.1,End Customer Sub Segment,388,Strategic - Public Sector
6,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250701.20250830.1,End Customer Sub Segment,408,Strategic - Federal Government
7,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250831.20260120.1,End Customer Sub Segment,385,Upper Majors - Education
8,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250831.20260120.1,End Customer Sub Segment,383,Upper Majors - Commercial
9,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250831.20260120.1,End Customer Sub Segment,388,Strategic - Public Sector


In [28]:
"""4.2 Option 1: Pivot territory data into a leader-level summary.

IncentiveTerritoryDetail stores territory dimensions as rows (DomainName/DomainValueName).
Pivot to get one row per leader per territory with SubRegion, Area, etc. as columns.
"""

# Show what DomainName values exist
logger.info("DomainName values in Option 1 results:")
domain_counts = df_option1["DomainName"].value_counts()
for domain, cnt in domain_counts.items():
    print(f"  {domain}: {cnt} rows")

# Pivot: create one row per leader+territory with domain values as columns
df_opt1_pivot = (
    df_option1
    .pivot_table(
        index=["FullName", "EmployeeAlias", "PlanID", "IncentiveTerritoryName"],
        columns="DomainName",
        values="DomainValueName",
        aggfunc="first",
    )
    .reset_index()
)
df_opt1_pivot.columns.name = None
logger.info("Pivoted to %d rows x %d columns", len(df_opt1_pivot), len(df_opt1_pivot.columns))
df_opt1_pivot.head(20)

INFO: DomainName values in Option 1 results:


  End Customer Sub Segment: 1967 rows
  Reported SubSegment: 1967 rows
  SubRegion: 589 rows
  Sales Unit: 94 rows
  Subsidiary: 23 rows
  Area: 21 rows


INFO: Pivoted to 622 rows x 10 columns


,FullName,EmployeeAlias,PlanID,IncentiveTerritoryName,Area,End Customer Sub Segment,Reported SubSegment,Sales Unit,SubRegion,Subsidiary
0,Agnes Heftberger,AHEFTBERGER,373,AHEFTBERGER.SSG.1.20250701.20260630.1,NaN,NaN,NaN,NaN,Germany,NaN
1,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.1.20250701.20250830.1,NaN,NaN,Strategic - Federal Government,NaN,Qatar,NaN
2,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.1.20250831.20260120.1,NaN,NaN,Majors Growth - Public Sector,NaN,Qatar,NaN
3,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250701.20250830.1,NaN,Upper Majors - Government,NaN,NaN,Qatar,NaN
4,Ahmad El Dandachi,AHMADEL,370,AHMADEL.SSG.2.20250831.20260120.1,NaN,Strategic - Commercial,NaN,NaN,Qatar,NaN
5,Ahmed Hamzawy,HAMZAWYAHMED,284,HAMZAWYAHMED.SSG.1.20250701.20250811.1,NaN,NaN,NaN,NaN,United Arab Emirates,NaN
6,Alaeddine Alaeddine,ALAKALA,959,ALAKALA.SSG.1.20250701.20260630.1,NaN,Upper Majors - Education,NaN,NaN,Kuwait,NaN
7,Alaeddine Alaeddine,ALAKALA,959,ALAKALA.SSG.2.20250701.20260630.1,NaN,NaN,Majors Growth - Commercial,NaN,Kuwait,NaN
8,Alon Haimovich,ALHAIMOV,370,ALHAIMOV.SSG.1.20250701.20260630.1,NaN,NaN,NaN,NaN,Israel,NaN
9,Amr Kamel,AMRK,370,AMRK.SSG.1.20250701.20260630.1,NaN,Digital Natives,NaN,NaN,United Arab Emirates,NaN


In [29]:
"""4.3 Option 1 summary statistics."""

print("OPTION 1 -- Territory Hierarchy Results (Pivoted)")
print("=" * 60)
print(f"Total territory assignments:  {len(df_opt1_pivot):,}")
print(f"Unique leaders:               {df_opt1_pivot['FullName'].nunique()}")
terr_col = "IncentiveTerritoryName"
print(f"Unique territories:           {df_opt1_pivot[terr_col].nunique()}")
# Show domain columns available
domain_cols = [c for c in df_opt1_pivot.columns if c not in
               ["FullName", "EmployeeAlias", "PlanID", terr_col]]
for col in domain_cols:
    n_unique = df_opt1_pivot[col].nunique()
    print(f"  Unique {col}: {n_unique}")
print(f"\nAvg territories per leader:   {len(df_opt1_pivot) / df_opt1_pivot['FullName'].nunique():.1f}")

# SubRegion distribution (key for teammate's question)
if "SubRegion" in df_opt1_pivot.columns:
    print(f"\nSubRegion distribution (top 15):")
    print(df_opt1_pivot["SubRegion"].value_counts().head(15).to_string())

OPTION 1 -- Territory Hierarchy Results (Pivoted)
Total territory assignments:  622
Unique leaders:               94
Unique territories:           622
  Unique Area: 9
  Unique End Customer Sub Segment: 17
  Unique Reported SubSegment: 17
  Unique Sales Unit: 11
  Unique SubRegion: 58
  Unique Subsidiary: 1

Avg territories per leader:   6.6

SubRegion distribution (top 15):
SubRegion
Nigeria                         16
Bahrain and Oman                16
Egypt                           16
Qatar                           14
Central and Caribbean Region    14
Multi Country Africa            14
Kenya                           14
Morocco                         14
United Arab Emirates            13
Saudi Arabia                    13
Kuwait                          12
Switzerland                     12
Mexico                          12
Ukraine                         12
Hungary                         12


In [30]:
"""4.4 Sample: Show a few leaders with their territory mapping."""

# Pick 3 leaders and show their full territory profile
sample_leaders = df_opt1_pivot["FullName"].unique()[:3]
for leader in sample_leaders:
    leader_rows = df_opt1_pivot[df_opt1_pivot["FullName"] == leader]
    print(f"\n{'='*60}")
    print(f"Leader: {leader}")
    print(f"  Alias: {leader_rows['EmployeeAlias'].iloc[0]}")
    print(f"  PlanID: {leader_rows['PlanID'].iloc[0]}")
    print(f"  Territory count: {len(leader_rows)}")
    domain_cols = [c for c in leader_rows.columns if c not in
                   ["FullName", "EmployeeAlias", "PlanID", "IncentiveTerritoryName"]]
    for _, r in leader_rows.head(5).iterrows():
        print(f"  Territory: {r['IncentiveTerritoryName']}")
        for dc in domain_cols:
            if pd.notna(r.get(dc)):
                print(f"    {dc}: {r[dc]}")
    if len(leader_rows) > 5:
        print(f"  ... and {len(leader_rows) - 5} more territories")


Leader: Agnes Heftberger
  Alias: AHEFTBERGER
  PlanID: 373
  Territory count: 1
  Territory: AHEFTBERGER.SSG.1.20250701.20260630.1
    SubRegion: Germany

Leader: Ahmad El Dandachi
  Alias: AHMADEL
  PlanID: 370
  Territory count: 4
  Territory: AHMADEL.SSG.1.20250701.20250830.1
    Reported SubSegment: Strategic - Federal Government
    SubRegion: Qatar
  Territory: AHMADEL.SSG.1.20250831.20260120.1
    Reported SubSegment: Majors Growth - Public Sector
    SubRegion: Qatar
  Territory: AHMADEL.SSG.2.20250701.20250830.1
    End Customer Sub Segment: Upper Majors - Government
    SubRegion: Qatar
  Territory: AHMADEL.SSG.2.20250831.20260120.1
    End Customer Sub Segment: Strategic - Commercial
    SubRegion: Qatar

Leader: Ahmed Hamzawy
  Alias: HAMZAWYAHMED
  PlanID: 284
  Territory count: 1
  Territory: HAMZAWYAHMED.SSG.1.20250701.20250811.1
    SubRegion: United Arab Emirates


---
## 5. Option 2: Account Assignments Approach (Filtered)

This is the **teammate's original approach**, but with proper filtering to reduce noise:
- Filter on `AssignmentType` to get primary assignments only
- Derive subregion/subsegment from the TPID mappings

**Why this approach can be noisy:**
- `employeeaccountassignments` reflects compensation-level account touchpoints
- Leaders may inherit SMEC TPIDs from their reports' rollups
- Without filtering, you get accounts the leader's comp touches, not accounts they own

In [31]:
"""5.1 Check available AssignmentType and RoleType values -- key filter fields.

The teammate's problem: EmployeeAccountAssignments shows SMEC TPIDs for
leaders who do not own SMEC accounts. Let us explore the filter columns.
"""

query_assignment_types = f"""
EmployeeAccountAssignments
| where DataFiscalYearID == {FISCAL_YEAR}
| where IsCurrent == true
| summarize Count=count() by AssignmentType, RoleType
| order by Count desc
"""

logger.info("Exploring AssignmentType x RoleType in EmployeeAccountAssignments...")
df_assignment_types = run_kql(query_assignment_types)
print("AssignmentType x RoleType distribution:")
print(df_assignment_types.to_string(index=False))

INFO: Exploring AssignmentType x RoleType in EmployeeAccountAssignments...
INFO: Query returned 71 rows x 3 columns.


AssignmentType x RoleType distribution:
AssignmentType RoleType    Count
MultiTerritory    Other 38683648
MultiTerritory      CSU  7520223
MultiTerritory     SMEC  7293136
          Area    Other  5826089
MultiTerritory      M&O  5695095
MultiTerritory      STU  5247580
MultiTerritory   OCP-SW  2718108
    Subsidiary      M&O  1529682
    Subsidiary      CSU  1508016
          Area     SMEC  1289475
     SubRegion      M&O  1256846
          Area       ES  1151137
     SubRegion     SMEC  1139561
    Subsidiary    Other   935553
    SalesGroup       ES   911258
    Subsidiary      STU   717012
     SalesUnit      CSU   530933
          Area      M&O   521443
          Area      CSU   462024
    SalesGroup    Other   351750
MultiTerritory Industry   340610
          Area      STU   319544
       Account      CSU   312396
       Account    Other   305247
       Account   OCP-BW   262610
     SubRegion    Other   256746
     SubRegion      CSU   241449
     SubRegion      STU   184012
   

In [32]:
"""5.2 Option 2 query -- map leaders to territories via EmployeeAccountAssignments.

This is the teammate's original approach. We run it TWO ways:
  a) UNFILTERED (all AssignmentTypes) -- shows the noise problem
  b) FILTERED to direct assignments only (Account, SalesTerritory)
"""

query_option2_all = f"""
let LeaderPlanIDs = dynamic([{plan_id_filter}]);
let Leaders =
    Participation
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
    | where PlanID in (LeaderPlanIDs)
    | join kind=inner (
        Person
        | where DataFiscalYearID == {FISCAL_YEAR}
        | where IsCurrent == true
        | project PersonnelNumber, FullName, EmployeeAlias
    ) on PersonnelNumber
    | project PersonnelNumber, FullName, EmployeeAlias, PlanID, ParticipationID;
Leaders
| join kind=inner (
    EmployeeAccountAssignments
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
) on ParticipationID
| summarize
    TotalRows = count(),
    UniqueTpids = dcount(TPID),
    AssignmentTypes = make_set(AssignmentType),
    RoleTypes = make_set(RoleType)
    by FullName, EmployeeAlias, PlanID
| order by TotalRows desc
"""

logger.info("Running Option 2a: UNFILTERED account assignments for OU leaders")
df_option2_summary = run_kql(query_option2_all)
logger.info("Leaders with account assignments: %d", len(df_option2_summary))
print(f"Top 15 leaders by total assignment rows (shows the noise):")
print(df_option2_summary.head(15).to_string(index=False))

INFO: Running Option 2a: UNFILTERED account assignments for OU leaders
INFO: Query returned 96 rows x 7 columns.
INFO: Leaders with account assignments: 96


Top 15 leaders by total assignment rows (shows the noise):
            FullName EmployeeAlias  PlanID  TotalRows  UniqueTpids                    AssignmentTypes RoleTypes
     Tammy Posnikoff       TBRANDT    1341     429626       213849                        [SubRegion]    [SMEC]
     Rodrigo Caserta      RCASERTA     374     124051       123083                   [MultiTerritory]    [SMEC]
       Travis Walter        466221     374     124051       123083                        [SubRegion]    [SMEC]
       Clare Barclay      CLARECUR     373      75637        75916                   [MultiTerritory]   [Other]
   Sharon Schoenborn      SSCHOENB     374      54621        54707                   [MultiTerritory]    [SMEC]
Melanie Sharpe Nseir       MSHARPE    1341      32489        32567                        [SubRegion]    [SMEC]
        Rachel Bondi        RCLARK     374      32489        32567                   [MultiTerritory]    [SMEC]
         Mark Chaban      MARKCHAA     373   

In [33]:
"""5.3 Option 2b: Show the SMEC noise problem explicitly.

For a sample leader, show what happens when we look at SMEC RoleType
vs non-SMEC -- this is exactly the teammate's reported issue.
"""

# Pick the leader with the most SMEC noise
query_smec_example = f"""
let LeaderPlanIDs = dynamic([{plan_id_filter}]);
let Leaders =
    Participation
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
    | where PlanID in (LeaderPlanIDs)
    | join kind=inner (
        Person
        | where DataFiscalYearID == {FISCAL_YEAR}
        | where IsCurrent == true
        | project PersonnelNumber, FullName, EmployeeAlias
    ) on PersonnelNumber
    | project PersonnelNumber, FullName, EmployeeAlias, PlanID, ParticipationID;
Leaders
| join kind=inner (
    EmployeeAccountAssignments
    | where DataFiscalYearID == {FISCAL_YEAR}
    | where IsCurrent == true
) on ParticipationID
| summarize
    TotalRows = count(),
    UniqueTpids = dcount(TPID)
    by FullName, EmployeeAlias, AssignmentType, RoleType
| order by FullName asc, TotalRows desc
"""

logger.info("Breaking down assignments by AssignmentType x RoleType per leader")
df_opt2_breakdown = run_kql(query_smec_example)

# Show top leaders with the most assignment types
print("Assignment breakdown for leaders with the most rows:")
print("(This shows why SMEC TPIDs appear for non-SMEC leaders)\n")

top_leaders = df_option2_summary.head(5)["FullName"].tolist()
for leader in top_leaders:
    rows = df_opt2_breakdown[df_opt2_breakdown["FullName"] == leader]
    total = rows["TotalRows"].sum()
    print(f"  {leader} -- {total:,} total assignment rows:")
    for _, r in rows.iterrows():
        print(f"    {r['AssignmentType']:>20} / {r['RoleType']:<10} "
              f"= {r['TotalRows']:>8,} rows, {r['UniqueTpids']:>8,} TPIDs")
    print()

INFO: Breaking down assignments by AssignmentType x RoleType per leader
INFO: Query returned 101 rows x 6 columns.


Assignment breakdown for leaders with the most rows:
(This shows why SMEC TPIDs appear for non-SMEC leaders)

  Tammy Posnikoff -- 429,626 total assignment rows:
               SubRegion / SMEC       =  429,626 rows,  213,849 TPIDs

  Rodrigo Caserta -- 124,051 total assignment rows:
          MultiTerritory / SMEC       =  124,051 rows,  123,083 TPIDs

  Travis Walter -- 124,051 total assignment rows:
               SubRegion / SMEC       =  124,051 rows,  123,083 TPIDs

  Clare Barclay -- 75,637 total assignment rows:
          MultiTerritory / Other      =   75,637 rows,   75,916 TPIDs

  Sharon Schoenborn -- 54,621 total assignment rows:
          MultiTerritory / SMEC       =   54,621 rows,   54,707 TPIDs



In [34]:
"""5.4 Option 2 summary statistics."""

print("OPTION 2 -- Account Assignments Results (Summary)")
print("=" * 60)
total_rows = df_option2_summary["TotalRows"].sum()
total_tpids = df_option2_summary["UniqueTpids"].sum()
print(f"Leaders with assignments:     {len(df_option2_summary)}")
print(f"Total assignment rows:        {total_rows:,}")
print(f"Sum of unique TPIDs:          {total_tpids:,}")
print(f"Avg rows per leader:          {total_rows / max(len(df_option2_summary), 1):,.0f}")
print(f"Avg TPIDs per leader:         {total_tpids / max(len(df_option2_summary), 1):,.0f}")

# How many leaders have SMEC in their RoleTypes?
smec_leaders = df_option2_summary[
    df_option2_summary["RoleTypes"].astype(str).str.contains("SMEC")
]
print(f"\nLeaders with SMEC assignments: {len(smec_leaders)} of {len(df_option2_summary)}")
print(f"  (These are the 'noisy' ones the teammate flagged)")

OPTION 2 -- Account Assignments Results (Summary)
Leaders with assignments:     96
Total assignment rows:        1,085,200
Sum of unique TPIDs:          849,163
Avg rows per leader:          11,304
Avg TPIDs per leader:         8,845

Leaders with SMEC assignments: 8 of 96
  (These are the 'noisy' ones the teammate flagged)


---
## 6. Compare Both Approaches Side-by-Side

### 6.1 Aggregate Comparison

Compare row counts, unique leaders, and territory counts to see where the approaches diverge.

In [35]:
"""6.1 Side-by-side aggregate comparison of both approaches."""

opt1_leaders = df_opt1_pivot["FullName"].nunique()
opt2_leaders = df_option2_summary["FullName"].nunique()
opt1_rows = len(df_opt1_pivot)
opt2_rows = int(df_option2_summary["TotalRows"].sum())
opt1_avg = opt1_rows / max(opt1_leaders, 1)
opt2_avg = opt2_rows / max(opt2_leaders, 1)

print("COMPARISON: Option 1 (Territory Hierarchy) vs Option 2 (Account Assignments)")
print("=" * 75)
print(f"{'Metric':<40} {'Option 1':>15} {'Option 2':>15}")
print("-" * 75)
print(f"{'Unique leaders found':<40} {opt1_leaders:>15,} {opt2_leaders:>15,}")
print(f"{'Total output rows':<40} {opt1_rows:>15,} {opt2_rows:>15,}")
print(f"{'Avg rows per leader':<40} {opt1_avg:>15.1f} {opt2_avg:>15,.0f}")
print(f"{'Data granularity':<40} {'Territory':>15} {'TPID':>15}")
print(f"{'SMEC noise problem':<40} {'None':>15} {'8 leaders':>15}")
print("=" * 75)
print()
print(f"Option 2 produces {opt2_rows / max(opt1_rows, 1):,.0f}x MORE rows than Option 1.")
print()
print("ROOT CAUSE of teammate's SMEC noise:")
print("  EmployeeAccountAssignments maps leaders to ALL TPIDs that")
print("  touch their comp at any level (MultiTerritory, SubRegion, etc).")
print("  OU leaders with SMEC-related plans inherit hundreds of thousands")
print("  of SMEC TPIDs through rollup assignments.")

# Show which leaders are in Option 1 but missing from Option 2 (or vice versa)
opt1_names = set(df_opt1_pivot["FullName"].unique())
opt2_names = set(df_option2_summary["FullName"].unique())
only_opt1 = opt1_names - opt2_names
only_opt2 = opt2_names - opt1_names

print(f"\nLeaders only in Option 1 (no account assignments):  {len(only_opt1)}")
if only_opt1:
    for name in sorted(only_opt1)[:5]:
        print(f"  - {name}")
    if len(only_opt1) > 5:
        print(f"  ... and {len(only_opt1) - 5} more")

print(f"Leaders only in Option 2 (no territory mapping):    {len(only_opt2)}")
if only_opt2:
    for name in sorted(only_opt2)[:5]:
        print(f"  - {name}")
    if len(only_opt2) > 5:
        print(f"  ... and {len(only_opt2) - 5} more")

COMPARISON: Option 1 (Territory Hierarchy) vs Option 2 (Account Assignments)
Metric                                          Option 1        Option 2
---------------------------------------------------------------------------
Unique leaders found                                  94              95
Total output rows                                    622       1,085,200
Avg rows per leader                                  6.6          11,423
Data granularity                               Territory            TPID
SMEC noise problem                                  None       8 leaders

Option 2 produces 1,745x MORE rows than Option 1.

ROOT CAUSE of teammate's SMEC noise:
  EmployeeAccountAssignments maps leaders to ALL TPIDs that
  touch their comp at any level (MultiTerritory, SubRegion, etc).
  OU leaders with SMEC-related plans inherit hundreds of thousands
  of SMEC TPIDs through rollup assignments.

Leaders only in Option 1 (no account assignments):  0
Leaders only in Option 2 (no

---
## 7. Final Recommendation

### Summary of Findings

In [36]:
"""7.1 Final recommendation summary."""

print()
print("=" * 70)
print("FINAL RECOMMENDATION")
print("=" * 70)
print("""
  APPROACH           USE WHEN
  -----------------  --------------------------------------------------
  Option 1           Mapping leaders to their OWNED territory
  (ParticipationTe-  (structural org/territory question)
   rritory + Incen-  "What territory does this leader own?"
   tiveTerritoryDe-  Returns: SubRegion, Area, Sales Unit, Subsidiary,
   tail)             End Customer Sub Segment
  ** RECOMMENDED **  622 rows for 94 leaders (avg 6.6 per leader)

  Option 2           Finding which accounts touch a leader's comp
  (EmployeeAccount-  (compensation/quota question)
   Assignments)      "What TPIDs roll up to this leader?"
                     1,085,200 rows for 95 leaders (avg 11,423 per leader)
  ** NOT FOR         8 leaders have SMEC noise (up to 429K rows each)
     TERRITORY **

KEY INSIGHT:
  EmployeeAccountAssignments answers: "What TPIDs touch this person's comp?"
  ParticipationTerritory answers:     "What territory does this person OWN?"

  For mapping OU leaders to territories, you want OWNERSHIP -- use Option 1.
  The join path is:
    Participation (PlanID filter)
    -> ParticipationTerritory (on ParticipationID)
    -> IncentiveTerritoryDetail (on IncentiveTerritoryID)

ROOT CAUSE OF TEAMMATE'S SMEC ISSUE:
  Leaders on plans like 'GPS Field OU/Segment SME&C Leader' (PlanID 1341)
  get SubRegion-level SMEC assignments in EmployeeAccountAssignments.
  These are comp rollup rows, not direct territory ownership.
  Example: Tammy Posnikoff has 429,626 SMEC TPID rows.
""")

print(f"Results for FY{FISCAL_YEAR}:")
print(f"  Option 1: {len(df_opt1_pivot):,} territory rows, {df_opt1_pivot['FullName'].nunique()} leaders")
print(f"  Option 2: {int(df_option2_summary['TotalRows'].sum()):,} TPID rows, {len(df_option2_summary)} leaders")
print("=" * 70)


FINAL RECOMMENDATION

  APPROACH           USE WHEN
  -----------------  --------------------------------------------------
  Option 1           Mapping leaders to their OWNED territory
  (ParticipationTe-  (structural org/territory question)
   rritory + Incen-  "What territory does this leader own?"
   tiveTerritoryDe-  Returns: SubRegion, Area, Sales Unit, Subsidiary,
   tail)             End Customer Sub Segment
  ** RECOMMENDED **  622 rows for 94 leaders (avg 6.6 per leader)

  Option 2           Finding which accounts touch a leader's comp
  (EmployeeAccount-  (compensation/quota question)
   Assignments)      "What TPIDs roll up to this leader?"
                     1,085,200 rows for 95 leaders (avg 11,423 per leader)
  ** NOT FOR         8 leaders have SMEC noise (up to 429K rows each)
     TERRITORY **

KEY INSIGHT:
  EmployeeAccountAssignments answers: "What TPIDs touch this person's comp?"
  ParticipationTerritory answers:     "What territory does this person OWN?"

  For

---
## 8. Export Results

Export both result sets and the comparison to CSV for further analysis or sharing with the team.

In [37]:
"""8.1 Export results to CSV files in the notebooks directory."""

import os

output_dir = os.path.dirname(os.path.abspath("__file__"))

# Export Option 1 (recommended) -- pivoted territory mapping
opt1_path = os.path.join(output_dir, f"ou_leaders_territory_hierarchy_FY{FISCAL_YEAR}.csv")
df_opt1_pivot.to_csv(opt1_path, index=False)
logger.info("Option 1 exported: %s (%d rows)", opt1_path, len(df_opt1_pivot))

# Export Option 2 summary (per-leader aggregation, not all 1M+ rows)
opt2_path = os.path.join(output_dir, f"ou_leaders_account_assignments_summary_FY{FISCAL_YEAR}.csv")
df_option2_summary.to_csv(opt2_path, index=False)
logger.info("Option 2 summary exported: %s (%d rows)", opt2_path, len(df_option2_summary))

# Export the leader roster
leaders_path = os.path.join(output_dir, f"ou_leaders_roster_FY{FISCAL_YEAR}.csv")
df_leaders.to_csv(leaders_path, index=False)
logger.info("Leader roster exported: %s (%d rows)", leaders_path, len(df_leaders))

print(f"\nExported files:")
print(f"  1. {opt1_path}")
print(f"  2. {opt2_path}")
print(f"  3. {leaders_path}")

INFO: Option 1 exported: c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_territory_hierarchy_FY2026.csv (622 rows)
INFO: Option 2 summary exported: c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_account_assignments_summary_FY2026.csv (96 rows)
INFO: Leader roster exported: c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_roster_FY2026.csv (109 rows)



Exported files:
  1. c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_territory_hierarchy_FY2026.csv
  2. c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_account_assignments_summary_FY2026.csv
  3. c:\Training\Microsoft\Copilot\kusto_app\notebooks\ou_leaders_roster_FY2026.csv


---
## 9. Valerie's Three-Approach Framework

Valerie (data expert) outlined three approaches for territory mapping, each with trade-offs:

| Approach | Pros | Cons |
|---|---|---|
| **EmployeeAccountAssignments** | Already at TPID level, matches SPM | No ComponentID -- gaps when plans have different territory roll-ups (~10% of sellers) |
| **ParticipationTerritory + IncentiveTerritoryDetail** | Has ComponentID for accurate roll-up | Heavy to join, territory at various domain levels, difficult to parse |
| **CreditDetails** | Most closely reflects actual crediting (accounts with actual/quota) | Not widely publicized, data is heavy, better suited for Synapse than Kusto |

Below we explore the sample queries Valerie shared alongside her notes.

In [5]:
"""9.1 Explore TerritoryAssignmentDefinition -- a table Valerie referenced
that we had not examined before.
"""

logger.info("--- TerritoryAssignmentDefinition schema ---")
df_tad_cols = run_kql("TerritoryAssignmentDefinition | getschema")
print("TerritoryAssignmentDefinition columns:")
for _, row in df_tad_cols.iterrows():
    print(f"  {row['ColumnName']} ({row['ColumnType']})")

print(f"\nTotal columns: {len(df_tad_cols)}")

INFO: --- TerritoryAssignmentDefinition schema ---
INFO: Query returned 30 rows x 4 columns.


TerritoryAssignmentDefinition columns:
  DataFiscalYearID (int)
  EffectiveStartDate (datetime)
  EffectiveEndDate (datetime)
  IsCurrent (bool)
  LastRefreshedDate (datetime)
  HashKey (string)
  MSSalesAccountID (int)
  CRMAccountID (string)
  PositionNumber (int)
  PersonnelNumber (int)
  SellerAssignmentGUID (string)
  AssignmentDefinitionGUID (string)
  AssignmentStartDate (datetime)
  AssignmentEndDate (datetime)
  AssignmentStatus (string)
  RolePlayed (string)
  RoleType (string)
  AssignmentType (string)
  AssignmentValue (string)
  JustificationCategory (string)
  JustificationDescription (string)
  DoNotAssignToAccount (bool)
  CreatedBy (string)
  CreatedOn (datetime)
  ModifiedBy (string)
  ModifiedOn (datetime)
  RequesterAlias (string)
  ReviewerAlias (string)
  AccountOrgType (string)
  EDLPModifiedDateTime (datetime)

Total columns: 30


In [6]:
"""9.2 Run Valerie's sample queries -- peek at 10 rows from each table.

These are the exact queries Valerie shared, limited to 10 rows for review.
"""

# 9.2a: TerritoryAssignmentDefinition
logger.info("--- TerritoryAssignmentDefinition sample ---")
df_tad_sample = run_kql("""
TerritoryAssignmentDefinition
| where IsCurrent and DataFiscalYearID == 2026
| limit 10
""")
print(f"TerritoryAssignmentDefinition: {len(df_tad_sample)} sample rows, {len(df_tad_sample.columns)} columns")
print(f"Columns: {list(df_tad_sample.columns)}")
df_tad_sample

INFO: --- TerritoryAssignmentDefinition sample ---
INFO: Query returned 10 rows x 30 columns.


TerritoryAssignmentDefinition: 10 sample rows, 30 columns
Columns: ['DataFiscalYearID', 'EffectiveStartDate', 'EffectiveEndDate', 'IsCurrent', 'LastRefreshedDate', 'HashKey', 'MSSalesAccountID', 'CRMAccountID', 'PositionNumber', 'PersonnelNumber', 'SellerAssignmentGUID', 'AssignmentDefinitionGUID', 'AssignmentStartDate', 'AssignmentEndDate', 'AssignmentStatus', 'RolePlayed', 'RoleType', 'AssignmentType', 'AssignmentValue', 'JustificationCategory', 'JustificationDescription', 'DoNotAssignToAccount', 'CreatedBy', 'CreatedOn', 'ModifiedBy', 'ModifiedOn', 'RequesterAlias', 'ReviewerAlias', 'AccountOrgType', 'EDLPModifiedDateTime']


,DataFiscalYearID,EffectiveStartDate,EffectiveEndDate,IsCurrent,LastRefreshedDate,HashKey,MSSalesAccountID,CRMAccountID,PositionNumber,PersonnelNumber,SellerAssignmentGUID,AssignmentDefinitionGUID,AssignmentStartDate,AssignmentEndDate,AssignmentStatus,RolePlayed,RoleType,AssignmentType,AssignmentValue,JustificationCategory,JustificationDescription,DoNotAssignToAccount,CreatedBy,CreatedOn,ModifiedBy,ModifiedOn,RequesterAlias,ReviewerAlias,AccountOrgType,EDLPModifiedDateTime
0,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,897542f4bee392a34176aa2d7dfc620c7a9125c314d8d4...,6032459,1-13FGBN0,<NA>,6257780,bc81192f-3963-f011-8dc9-6045bd2dad49,355d349f-3581-49b3-bb4a-e9720dbef59d,2025-07-01 00:00:00+00:00,2026-06-30 00:00:00+00:00,ACTIVE,Customer Success Unit,CSU,Account,AYESA INGENIERIA DE FUTURO SOCIEDAD ANONIMA,,,False,a-alrizea,2025-07-17 18:09:39.437000+00:00,a-alrizea,2025-07-17 18:44:34.103000+00:00,a-alrizea,,End Customer,2026-02-23 09:08:42.962751+00:00
1,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,e77a1189eb753d68130f41a895c740874eec43e4703ea6...,<NA>,7-3FAEDH3L2Z,<NA>,207120,02ae9c5e-3995-f011-8e60-00224848eff0,c714ed85-af8d-f011-b480-00224848eff0,2025-07-01 00:00:00+00:00,2026-06-30 00:00:00+00:00,ACTIVE,Build With,OCP-BW,Account,Dynatrace LLC,,,False,annakasp,2025-09-19 09:16:50.647000+00:00,annakasp,2025-09-19 09:43:09.327000+00:00,annakasp,,Partner,2026-02-23 09:08:42.962751+00:00
2,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,856be2a88b7f112213b56165d2d5510acd7fc84431c8f6...,<NA>,7-3FAEHWXBNU,<NA>,604457,1f921a00-5807-f111-a69a-6045bd320818,4d1aed85-af8d-f011-b480-00224848eff0,2026-01-15 00:00:00+00:00,9999-12-31 00:00:00+00:00,ACTIVE,Technical,OCP-BW,Account,PROART Consulting,"People Change (i.e. new hire, role change)",,False,v-astunkel,2026-02-11 14:43:19.637000+00:00,v-astunkel,2026-02-17 22:29:17.593000+00:00,v-astunkel,,Partner,2026-02-23 09:08:42.962751+00:00
3,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,1c73317dd4f1151a7d54126c30854561038d6ba444dc13...,82739918,9-33PWKTS723,<NA>,6285467,687435b4-455e-f011-8dc9-6045bd2dad49,a28287c5-d38c-4e59-81eb-201934adb300,2025-07-01 00:00:00+00:00,2025-09-14 00:00:00+00:00,EXPIRED,Customer Success Unit,CSU,Account,SILAE,,,False,petartodorov,2025-07-11 10:56:40.503000+00:00,SPMAutoExpiryUser,2025-10-09 04:05:35.873000+00:00,SPMAutoExpiryUser,,End Customer,2026-02-23 09:08:42.962751+00:00
4,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,d214e478fc47735c934f9d70057faf358c5e17b3580680...,1858710,9-34WGO4ZLJT,<NA>,6471879,3c197b7f-3ab5-f011-8194-6045bd2dad49,13b4ebd7-1d73-4535-85c6-18d086d8bcaf,2025-09-11 00:00:00+00:00,2026-06-30 00:00:00+00:00,ACTIVE,Business Management,Other,Account,CONSTRUCTION INDUSTRY LONG SERVICE LEAVE BOARD,Organization Change,Others,False,crmiri,2025-10-30 02:45:35.643000+00:00,v-sargsingh,2025-10-30 12:10:43.133000+00:00,v-sargsingh,,End Customer,2026-02-23 09:08:42.962751+00:00
5,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,0bed0e0f36ff5f0d8d5d6dcfd0bfeacf0416da2a5f618a...,12233131,1-FUV4FP,<NA>,1308469,b90265f3-37f6-f011-832e-6045bd320818,7359225b-1b56-470e-bf56-9ca8960c62e9,2025-12-05 00:00:00+00:00,2026-06-30 00:00:00+00:00,ACTIVE,Customer Success Unit,CSU,Account,Makemy Trip India Pvt Ltd,"People Change (i.e. new hire, role change)",,False,v-ddix,2026-01-20 19:41:07.767000+00:00,v-ddix,2026-01-20 19:41:35.003000+00:00,v-ddix,,End Customer,2026-02-23 09:08:42.962751+00:00
6,2026,2026-02-23 11:30:26+00:00,3000-12-31 00:00:00+00:00,True,2026-02-23 11:30:13.182706+00:00,823fd3acc534841f8625fc57b7fe109ceed2972b0fdddf...,<NA>,PAR-2071-TVM,<NA>,1335347,63bf7267-810b-f111-a69a-6045bd320818,db237db3-51be-4000-a5e1-23c4cf29cac9,2026-01-01 00:00:00+00:00,9999-12-31

In [7]:
"""9.3 Check if CreditDetails table exists and explore its schema."""

# First check if CreditDetails exists
logger.info("--- Checking for CreditDetails table ---")
df_credit_check = run_kql("""
.show tables
| where TableName has_cs 'Credit'
| project TableName
| order by TableName asc
""")
print("Tables containing 'Credit':")
for _, row in df_credit_check.iterrows():
    print(f"  {row['TableName']}")

# If it exists, get schema
if any(df_credit_check["TableName"].str.contains("CreditDetail", case=False)):
    credit_table = df_credit_check[
        df_credit_check["TableName"].str.contains("CreditDetail", case=False)
    ]["TableName"].iloc[0]
    logger.info("--- %s schema ---", credit_table)
    df_credit_cols = run_kql(f"{credit_table} | getschema")
    print(f"\n{credit_table} columns:")
    for _, row in df_credit_cols.iterrows():
        print(f"  {row['ColumnName']} ({row['ColumnType']})")
    print(f"\nTotal columns: {len(df_credit_cols)}")
else:
    logger.warning("No CreditDetails table found -- Valerie noted this is better suited for Synapse.")

INFO: --- Checking for CreditDetails table ---
INFO: Query returned 0 rows x 1 columns.


Tables containing 'Credit':


In [8]:
"""9.4 Compare key columns across all three approaches.

Valerie's key insight: EmployeeAccountAssignments has NO ComponentID,
while ParticipationTerritory DOES. This matters when plans have different
territory roll-ups per component (~10% of sellers).
"""

print("KEY COLUMN COMPARISON ACROSS APPROACHES")
print("=" * 70)
print()
print("Join Key Columns:")
print(f"  {'Column':<30} {'EAA':^10} {'PT+ITD':^10} {'TAD':^10}")
print(f"  {'-'*30} {'-'*10} {'-'*10} {'-'*10}")

# Check which key columns exist in each table
eaa_cols = set(df_tad_cols["ColumnName"].tolist()) if "df_eaa_cols" not in dir() else set()
# We need to re-fetch schemas since kernel restarted
df_eaa_cols2 = run_kql("EmployeeAccountAssignments | getschema")
df_pt_cols2 = run_kql("ParticipationTerritory | getschema")
df_tad_cols2 = df_tad_cols  # already fetched

eaa_set = set(df_eaa_cols2["ColumnName"].tolist())
pt_set = set(df_pt_cols2["ColumnName"].tolist())
tad_set = set(df_tad_cols2["ColumnName"].tolist())

key_cols = [
    "ParticipationID", "PersonnelNumber", "PlanID", "ComponentID",
    "TPID", "IncentiveTerritoryID", "IncentiveTerritoryName",
    "AssignmentType", "RoleType", "Role",
    "SubRegion", "SubSegment", "Area",
    "DomainName", "DomainValueName",
]

for col in key_cols:
    in_eaa = "Y" if col in eaa_set else "-"
    in_pt = "Y" if col in pt_set else "-"
    in_tad = "Y" if col in tad_set else "-"
    print(f"  {col:<30} {in_eaa:^10} {in_pt:^10} {in_tad:^10}")

print()
print("NOTE: EmployeeAccountAssignments has NO ComponentID.")
print("      ParticipationTerritory HAS ComponentID.")
print("      This is the ~10% gap Valerie flagged.")

KEY COLUMN COMPARISON ACROSS APPROACHES

Join Key Columns:
  Column                            EAA       PT+ITD      TAD    
  ------------------------------ ---------- ---------- ----------


INFO: Query returned 21 rows x 4 columns.
INFO: Query returned 15 rows x 4 columns.


  ParticipationID                    Y          Y          -     
  PersonnelNumber                    Y          Y          Y     
  PlanID                             -          -          -     
  ComponentID                        -          Y          -     
  TPID                               Y          -          -     
  IncentiveTerritoryID               -          Y          -     
  IncentiveTerritoryName             -          Y          -     
  AssignmentType                     Y          -          Y     
  RoleType                           Y          -          Y     
  Role                               Y          -          -     
  SubRegion                          -          -          -     
  SubSegment                         -          -          -     
  Area                               -          -          -     
  DomainName                         -          -          -     
  DomainValueName                    -          -          -     

NOTE: Emp

---
## 10. GPS Territory Roll-Up Logic (from Valerie's Reference Video)

Source: "WWIC Ed - MintX Territory Roll-ups in GPS" (March 11, 2025, presented by Monika Siroha)

### Key Concepts

**1. GPS vs Commercial Sellers:**
- Commercial sellers are assigned at the **TPID level** (account ID in SPM)
- GPS sellers are assigned at the **Partner One ID level** (unique partner ID)
- GPS uses a dedicated SPM view: "Partner Account Assignments" (vs "Seller Assignments" for commercial)
- GPS role type in SPM: **OCP-BW** (OCP Build With; OCP = old name for GPS)

**2. Management Level (critical for territory scope):**
- Defines the **geographic scope of accountability** for a seller assigned to a partner
- Five levels processed by MintX:
  - **Global** -- worldwide accountability (core team, global partners)
  - **Area** -- area-level (e.g., ASEAN, RCR)
  - **Region** -- region in assigned area (multi-sub-area)
  - **Sub Region** -- sub-region in assigned area
  - **Local** -- local subsidiary only
- If multiple partners with different management levels: **highest level wins**
- Management level is set by the GPS team via the Managed Partner List (MPL)

**3. Territory Automation Source:**
- **OCP CRM** -- directs MintX to look at Partner Account Assignment view (87 plans, 2130 sellers)
- **Calc** -- directs MintX to commercial assignment view (3 plans, 88 sellers; e.g., Surface PDMs)
- Plans with OCP CRM can have partner influence metrics; plans with Calc cannot

**4. Planned Deployment Options:**
- **Partner One ID** -- lowest level, direct partner accountability
- **Geography** -- calculated from management level + assigned subsidiaries, then rolled up

**5. Geography Roll-Up Logic (2-step process):**
- **Step 1**: Determine geo coverage from management level + assigned subsidiary
  - Global -> Worldwide
  - Area -> Area of assigned subsidiary
  - Region -> Region of assigned subsidiary
  - Sub Region -> Sub-region of assigned subsidiary
  - Local -> Subsidiary itself
- **Step 2**: Match to planned deployment option
  - If deployment = sub-region and Step 1 = sub-region -> stays at sub-region
  - If deployment = area and Step 1 = subsidiary -> rolls UP to area
  - If deployment = subsidiary and Step 1 = area -> expands DOWN to all subsidiaries in that area

### Why This Matters for Yifei's Question
- OU leaders span both Commercial and GPS organizations
- GPS territory logic is fundamentally different (Partner ID vs TPID, management level, OCP-BW role type)
- The SMEC noise in `EmployeeAccountAssignments` likely comes from the commercial side
- GPS sellers use the OCP-BW role type in assignments -- this can be used as a filter
- The `TerritoryAssignmentDefinition` table has `RolePlayed` and `AccountOrgType` columns that may help distinguish GPS from commercial assignments

In [9]:
"""10.1 Validate GPS-specific concepts from Monica's presentation.

Check RoleType='OCP-BW' in both EAA and TAD, and look for
management-level related fields in TerritoryAssignmentDefinition.
"""

# 10.1a: OCP-BW role type distribution in EmployeeAccountAssignments
logger.info("--- OCP-BW (GPS) in EmployeeAccountAssignments ---")
df_ocpbw_eaa = run_kql(f"""
EmployeeAccountAssignments
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| where RoleType == 'OCP-BW'
| summarize
    Sellers = dcount(PersonnelNumber),
    TotalRows = count()
    by AssignmentType
| order by TotalRows desc
""")
print("OCP-BW (GPS) assignments in EmployeeAccountAssignments:")
print(df_ocpbw_eaa.to_string(index=False))

print()

# 10.1b: OCP-BW role type in TerritoryAssignmentDefinition
logger.info("--- OCP-BW (GPS) in TerritoryAssignmentDefinition ---")
df_ocpbw_tad = run_kql(f"""
TerritoryAssignmentDefinition
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| where RoleType == 'OCP-BW'
| summarize
    Sellers = dcount(PersonnelNumber),
    TotalRows = count(),
    RolesPlayed = make_set(RolePlayed)
    by AssignmentType
| order by TotalRows desc
""")
print("OCP-BW (GPS) assignments in TerritoryAssignmentDefinition:")
print(df_ocpbw_tad.to_string(index=False))

print()

# 10.1c: Check RolePlayed values (Monica mentioned 'Partner Business Management' is excluded)
logger.info("--- RolePlayed values for OCP-BW ---")
df_role_played = run_kql(f"""
TerritoryAssignmentDefinition
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| where RoleType == 'OCP-BW'
| summarize Count = count() by RolePlayed
| order by Count desc
""")
print("RolePlayed values (GPS OCP-BW):")
print(df_role_played.to_string(index=False))
print("\nNote: 'Partner Business Management' is excluded by MintX for RBI compensation.")

INFO: --- OCP-BW (GPS) in EmployeeAccountAssignments ---
INFO: Query returned 6 rows x 3 columns.
INFO: --- OCP-BW (GPS) in TerritoryAssignmentDefinition ---


OCP-BW (GPS) assignments in EmployeeAccountAssignments:
AssignmentType  Sellers  TotalRows
       Account     1938     262598
          Area      120        694
    Subsidiary      107        312
MultiTerritory      115        117
     SubRegion       36         69
        Region        1          1



INFO: Query returned 6 rows x 4 columns.
INFO: --- RolePlayed values for OCP-BW ---


OCP-BW (GPS) assignments in TerritoryAssignmentDefinition:
AssignmentType  Sellers  TotalRows                                                                    RolesPlayed
       Account     1966     282431 [Technical, Build With, Build With - Primary PDM, Partner Business Management]
          Area      128        714                           [Technical, Build With, Partner Business Management]
    Subsidiary      149        463                           [Build With, Technical, Partner Business Management]
     SubRegion      114        351                                                        [Build With, Technical]
MultiTerritory      162        194                           [Build With, Technical, Partner Business Management]
        Region        5          5                                                        [Technical, Build With]



INFO: Query returned 4 rows x 2 columns.


RolePlayed values (GPS OCP-BW):
                 RolePlayed  Count
                  Technical 235858
                 Build With  39417
   Build With - Primary PDM   8301
Partner Business Management    582

Note: 'Partner Business Management' is excluded by MintX for RBI compensation.


In [10]:
"""10.2 Check ParticipationTerritoryDataSource for automation source info.

Monica mentioned 'OCP CRM' vs 'Calc' as territory automation sources.
The ParticipationTerritoryDataSource table (found in our earlier discovery)
might contain this information.
"""

# Check if ParticipationTerritoryDataSource has automation source info
logger.info("--- ParticipationTerritoryDataSource schema ---")
df_ptds_cols = run_kql("ParticipationTerritoryDataSource | getschema")
print("ParticipationTerritoryDataSource columns:")
for _, row in df_ptds_cols.iterrows():
    print(f"  {row['ColumnName']} ({row['ColumnType']})")

print()

# Sample data
logger.info("--- ParticipationTerritoryDataSource sample ---")
df_ptds_sample = run_kql(f"""
ParticipationTerritoryDataSource
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| limit 10
""")
print(f"Sample: {len(df_ptds_sample)} rows, {len(df_ptds_sample.columns)} columns")
print(f"Columns: {list(df_ptds_sample.columns)}")

# Check if there's a TerritoryAutomationSource column or similar
auto_cols = [c for c in df_ptds_cols["ColumnName"].tolist()
             if "auto" in c.lower() or "source" in c.lower() or "type" in c.lower()]
if auto_cols:
    print(f"\nAutomation/source-related columns: {auto_cols}")
    for col in auto_cols:
        vals = run_kql(f"""
ParticipationTerritoryDataSource
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| summarize Count = count() by {col}
| order by Count desc
| limit 20
""")
        print(f"\n{col} values:")
        print(vals.to_string(index=False))

INFO: --- ParticipationTerritoryDataSource schema ---
INFO: Query returned 9 rows x 4 columns.
INFO: --- ParticipationTerritoryDataSource sample ---


ParticipationTerritoryDataSource columns:
  DataFiscalYearID (int)
  EffectiveStartDate (datetime)
  EffectiveEndDate (datetime)
  IsCurrent (bool)
  LastRefreshedDate (datetime)
  HashKey (string)
  ParticipationID (int)
  TerritoryDataSourceID (int)
  PersonnelNumber (int)



INFO: Query returned 10 rows x 9 columns.


Sample: 10 rows, 9 columns
Columns: ['DataFiscalYearID', 'EffectiveStartDate', 'EffectiveEndDate', 'IsCurrent', 'LastRefreshedDate', 'HashKey', 'ParticipationID', 'TerritoryDataSourceID', 'PersonnelNumber']

Automation/source-related columns: ['TerritoryDataSourceID']


INFO: Query returned 5 rows x 2 columns.



TerritoryDataSourceID values:
 TerritoryDataSourceID  Count
                     1  36537
                     0  12163
                     6   4051
                     2   2787
                     7   1852


In [12]:
"""10.3 Check TerritoryAutomationSource table for OCP CRM vs Calc mapping.

Monica described 'OCP CRM' vs 'Calc' as territory automation sources.
This reference table maps TerritoryDataSourceID to TerritoryDataSourceName.
"""

# Get all automation source names
logger.info("--- TerritoryAutomationSource values ---")
df_tas_values = run_kql(f"""
TerritoryAutomationSource
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| project TerritoryDataSourceID, TerritoryDataSourceName
| distinct *
| order by TerritoryDataSourceID asc
""")
print("Territory Automation Sources (reference table):")
print(df_tas_values.to_string(index=False))

print()

# Now join to ParticipationTerritoryDataSource to see distribution
logger.info("--- Distribution of automation sources across participations ---")
df_tas_dist = run_kql(f"""
ParticipationTerritoryDataSource
| where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
| join kind=inner (
    TerritoryAutomationSource
    | where IsCurrent and DataFiscalYearID == {FISCAL_YEAR}
    | project TerritoryDataSourceID, TerritoryDataSourceName
    | distinct *
) on TerritoryDataSourceID
| summarize Participations = count() by TerritoryDataSourceName
| order by Participations desc
""")
print("Automation source distribution across participations:")
print(df_tas_dist.to_string(index=False))

INFO: --- TerritoryAutomationSource values ---
INFO: Query returned 9 rows x 2 columns.
INFO: --- Distribution of automation sources across participations ---


Territory Automation Sources (reference table):
 TerritoryDataSourceID TerritoryDataSourceName
                     0                  Manual
                     1                     SPM
                     2                     GPS
                     3         Demand Response
                     4           Retail Stores
                     5                    MIC2
                     6                     UBI
                     7                     MSA
                     8                     IOT



INFO: Query returned 5 rows x 2 columns.


Automation source distribution across participations:
TerritoryDataSourceName  Participations
                    SPM           36537
                 Manual           12163
                    UBI            4051
                    GPS            2787
                    MSA            1852
